# 第3章 模型如何学习：损失函数与优化（学生版）

本实验只研究**训练过程**。40条记录全部由程序人工生成，并非物理测量或天文观测。
数据划分已经固定：30条训练记录用于本章；另外10条保留记录不参与计算或评价。
`loss` 始终指训练数据上的**半均方误差**，不代表新样本预测能力。

建议从本目录或上一层 `ch03` 目录启动。运行全部单元可得到完整基础流程。
学生任务写在文字单元中，不会中断基础代码；请另加单元作答。
目标是解释参数为何改变、学习率与特征尺度为何重要，以及一次更新和一轮数据遍历的差别。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "line_data.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "data" / "line_data.csv").exists():
    raise FileNotFoundError("请从 ch03 或 ch03/notebooks 目录运行")
plt.rcParams["axes.unicode_minus"] = False
import sys
sys.path.insert(0, str(ROOT.parent / "experiments"))
import common as _book_plot_setup


## 1. 先判断：数据、预测和参数

读取数据前，请写出直线模型的输入、目标和两个可学习参数。
读取后只选择 `split == 'train'`。`sample_id` 和 `split` 不作为预测特征。

In [2]:
table = pd.read_csv(ROOT / "data" / "line_data.csv")
train = table.loc[table["split"].eq("train")].copy()
x = train["x"].to_numpy()
y = train["y"].to_numpy()
print("训练样本数:", len(train))
print(train.head().to_string(index=False))

训练样本数: 30
sample_id         x         y split
     S001 -2.000000 -4.437937 train
     S002 -1.897436 -3.871418 train
     S004 -1.692308 -3.012038 train
     S006 -1.487179 -3.005563 train
     S008 -1.282051 -2.313978 train


In [3]:
def half_mse(x, y, theta):
    """Training half mean squared error for theta=[slope, intercept]."""
    prediction = theta[0] * x + theta[1]
    return float(np.mean((prediction - y) ** 2) / 2)

def gradient(x, y, theta):
    """Both gradient components are computed at the same old theta."""
    error = theta[0] * x + theta[1] - y
    return np.array([np.mean(error * x), np.mean(error)])

def gradient_descent(x, y, lr, steps, threshold=1e12):
    """Full batch GD; index 0 records initialization before any update."""
    theta = np.zeros(2, dtype=float)
    rows = [{"step": 0, "a": 0.0, "b": 0.0,
             "loss": half_mse(x, y, theta)}]
    stop = "maximum_updates"
    for step in range(1, steps + 1):
        theta = theta - lr * gradient(x, y, theta)
        loss = half_mse(x, y, theta)
        rows.append({"step": step, "a": float(theta[0]),
                     "b": float(theta[1]), "loss": loss})
        if not np.isfinite(loss) or loss > threshold:
            stop = "divergence_guard"
            break
    return pd.DataFrame(rows), stop

## 2. 单参数损失：方向与步长

此处 $J(w)=(w-3)^2/2$ 是人为设置的示意目标，与直线CSV无关。
从 $w=0$ 出发，先手算学习率0.5时的两次更新，再运行。
比较0.1、0.5、1.5和2.1；记录哪些轨迹跨过了最小点，以及跨过后是否收敛。

In [4]:
rates = [0.1, 0.5, 1.5, 2.1]
scalar_records = []
for rate in rates:
    w = 0.0
    for step in range(13):
        scalar_records.append({"rate": rate, "step": step,
                               "w": w, "loss": (w - 3) ** 2 / 2})
        w = w - rate * (w - 3)
scalar = pd.DataFrame(scalar_records)
print(scalar.loc[(scalar["rate"] == 0.5) & (scalar["step"] <= 3)]
      .to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 4))
for rate, group in scalar.groupby("rate"):
    ax.semilogy(group["step"], group["loss"], marker="o", label=str(rate))
ax.set(xlabel="Parameter updates", ylabel="Illustrative loss J(w)")
ax.legend(title="Learning rate")
ax.grid(alpha=0.2)
plt.show()

 rate  step     w     loss
  0.5     0 0.000 4.500000
  0.5     1 1.500 1.125000
  0.5     2 2.250 0.281250
  0.5     3 2.625 0.070312


/tmp/ipykernel_2763/661673413.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**解释任务A**：为什么学习率1.5会左右交替，却仍接近最小点？
把2.1改成2.0，先预测损失会怎样，再验证。不能把本题的学习率边界推广给所有目标函数。

图例有四个学习率，但0.5和1.5的损失曲线完全重合：二者到最小点的距离每步都减半，
仅参数偏差的符号变化不同。平方损失消除了符号，所以这不是漏画了一条曲线。

## 3. 同时学习斜率和截距

保持训练数据、初始参数和目标函数不变，采用全批量梯度下降。
每次同时计算两个偏导，再同时更新两个参数。第0步尚未进行更新。

In [5]:
history, stop_reason = gradient_descent(x, y, lr=0.2, steps=120)
print(history.loc[history["step"].isin([0, 1, 5, 20, 120])]
      .to_string(index=False))
print("停止原因:", stop_reason)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x, y, s=25, color="black", label="Synthetic train data")
grid = np.linspace(x.min(), x.max(), 100)
for step in [0, 1, 5, 20, 120]:
    row = history.loc[history["step"].eq(step)].iloc[0]
    axes[0].plot(grid, row["a"] * grid + row["b"], label=f"step {step}")
axes[0].set(xlabel="x (dimensionless)", ylabel="y (dimensionless)")
axes[0].legend(fontsize=8)
axes[1].semilogy(history["step"], history["loss"])
axes[1].set(xlabel="Parameter updates", ylabel="Training half MSE")
axes[1].grid(alpha=0.2)
plt.show()

 step        a        b     loss
    0 0.000000 0.000000 5.168790
    1 0.741376 0.232381 2.593654
    5 2.074561 0.673205 0.186735
   20 2.505119 0.850248 0.026649
  120 2.506900 0.853364 0.026641
停止原因: maximum_updates


/tmp/ipykernel_2763/2879194253.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
least_squares = LinearRegression().fit(train[["x"]], train["y"])
reference_theta = np.array([least_squares.coef_[0], least_squares.intercept_])
final_theta = history[["a", "b"]].iloc[-1].to_numpy()
print("梯度下降参数:", final_theta)
print("最小二乘参数:", reference_theta)
print("最小二乘训练半MSE:", half_mse(x, y, reference_theta))
print("参数最大绝对差:", np.max(np.abs(final_theta - reference_theta)))

梯度下降参数: [2.50689953 0.85336381]
最小二乘参数: [2.50689953 0.85336381]
最小二乘训练半MSE: 0.026641478543588153
参数最大绝对差: 1.1278755707166965e-12


`LinearRegression` 在这里采用常规最小二乘数值求解；它不默认执行本实验的梯度下降循环。
两种求解路径在同一目标上得到相近结果，有助于检查梯度实现。
**解释任务B**：生成公式的斜率和截距是2.5与0.8，为什么拟合值不必恰好等于它们？
为什么训练损失变小仍不能说明模型已经能处理新的观测？

## 4. 特征尺度：相同信息，不同更新过程

把唯一特征改写成 `x_large = 1000 * x`，不要把x与x_large同时作为两列特征。
标准化器只在训练记录上拟合。先预测直接沿用0.2学习率会发生什么。

In [7]:
x_large = 1000.0 * x
scale = StandardScaler().fit(x_large.reshape(-1, 1))
z = scale.transform(x_large.reshape(-1, 1)).ravel()
raw_fast, fast_stop = gradient_descent(x_large, y, 0.2, 120)
raw_small, _ = gradient_descent(x_large, y, 1e-7, 120)
standardized, _ = gradient_descent(z, y, 0.2, 120)
print("大尺度、原学习率:", raw_fast.tail(1).to_string(index=False), fast_stop)
print("大尺度、极小学习率:", raw_small.tail(1).to_string(index=False))
print("标准化:", standardized.tail(1).to_string(index=False))
az, bz = standardized[["a", "b"]].iloc[-1]
original_a = 1000.0 * az / scale.scale_[0]
original_b = bz - az * scale.mean_[0] / scale.scale_[0]
print("标准化模型换回原x坐标:", original_a, original_b)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(raw_small["step"], raw_small["loss"], label="Large x, rate 1e-7")
ax.semilogy(standardized["step"], standardized["loss"], label="Scaled x, rate 0.2")
ax.set(xlabel="Parameter updates", ylabel="Training half MSE")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

大尺度、原学习率:  step             a             b         loss
    2 -2.130363e+08 -18248.825236 3.260363e+22 divergence_guard
大尺度、极小学习率:  step       a       b     loss
  120 0.00258 0.00001 0.386909
标准化:  step        a        b     loss
  120 2.989026 1.161905 0.026641
标准化模型换回原x坐标: 2.506899531682334 0.8533638087743225


/tmp/ipykernel_2763/2017976773.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


损失超过$10^{12}$会触发保护性停止；这是发散保护，不是成功收敛。
**解释任务C**：极小学习率下为什么可能很快学到大致斜率，但截距仍接近0？
为什么直接比较标准化前后的斜率数字会产生误解？

In [8]:
def batch_descent(x, y, lr, epochs, batch_size, seed):
    """Same constant learning rate, with loss evaluated on all train rows.

    Epochs have an equal number of sample visits, not equal update counts.
    Loss evaluation itself is omitted from sample_visit accounting.
    """
    n = len(x)
    theta = np.zeros(2, dtype=float)
    rng = np.random.default_rng(seed)
    rows = [{"step": 0, "sample_visits": 0, "epoch_passes": 0.0,
             "a": 0.0, "b": 0.0, "loss": half_mse(x, y, theta)}]
    sample_visits = 0
    step = 0
    for epoch in range(epochs):
        order = np.arange(n) if batch_size == n else rng.permutation(n)
        for start in range(0, n, batch_size):
            index = order[start:start + batch_size]
            theta = theta - lr * gradient(x[index], y[index], theta)
            sample_visits += len(index)
            step += 1
            rows.append({"step": step, "sample_visits": sample_visits,
                         "epoch_passes": sample_visits / n,
                         "a": float(theta[0]), "b": float(theta[1]),
                         "loss": half_mse(x, y, theta)})
    return pd.DataFrame(rows)

## 5. 一次更新与一轮遍历

使用同一训练标准化特征z、相同初值0、相同常数学习率0.1和80轮遍历。
批量大小30、6、1分别对应每轮1、5、30次更新。这里控制的是样本遍历量，
不是更新次数或运行时间。每个图中纵坐标都由全部30条训练记录计算。

In [9]:
batch_histories = {}
for label, size in [("Full batch", 30), ("Mini batch", 6), ("SGD", 1)]:
    run = batch_descent(z, y, lr=0.1, epochs=80, batch_size=size, seed=42)
    batch_histories[label] = run
    print(label, run.tail(1).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, run in batch_histories.items():
    axes[0].semilogy(run["epoch_passes"], run["loss"], label=label)
    recent = run.loc[run["epoch_passes"] >= 70]
    axes[1].plot(recent["epoch_passes"], recent["loss"], label=label)
axes[0].set(xlabel="Sample visits / training size", ylabel="Training half MSE")
axes[1].set(xlabel="Sample visits / training size", ylabel="Training half MSE")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
plt.show()

Full batch  step  sample_visits  epoch_passes        a        b     loss
   80           2400          80.0 2.988373 1.161651 0.026642
Mini batch  step  sample_visits  epoch_passes        a        b     loss
  400           2400          80.0 2.991085 1.160417 0.026645
SGD  step  sample_visits  epoch_passes        a       b     loss
 2400           2400          80.0 2.998457 1.12565 0.027343


/tmp/ipykernel_2763/797781901.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**解释任务D**：把随机种子改为0、1、2，比较末步结果。更换批量大小后，
“80轮”为什么不再表示相同更新次数？观察随机更新的某一步使训练损失增加，
能否立刻判断梯度代码错误？实验包含额外的全训练损失记录，不适合直接比较训练速度。

## 6. 提交内容

提交两步手算、直线参数更新记录、学习率与尺度对照图，并回答任务A—D。
另用150—250字说明本章的停止条件、结果能支持的判断，以及尚未回答的泛化问题。
保留训练/保留划分，不利用保留记录调学习率或做本章的模型评价。